# PHASE 4: Modeling (Regression & Clustering)

Phần này ứng dụng Học máy (Machine Learning) để giải quyết hai bài toán kinh doanh chính.

In [ ]:
library(dplyr)
library(readr)
library(ggplot2)
library(randomForest)

### 1. Phân cụm (K-Means)

**Mô tả:** Thuật toán K-Means chia các phòng Airbnb thành 4 cụm (Clusters) dựa trên Giá, Số lượng đánh giá và Tọa độ địa lý.

**Kết luận:** Mô hình đã phân chia thị trường thành 4 phân khúc:
1. **Cụm 1:** Phòng giá rẻ, ít được review.
2. **Cụm 2:** Phòng tầm trung, mật độ dày, review cao.
3. **Cụm 3:** Phòng giá cao, vị trí đắc địa ở Manhattan.
4. **Cụm 4:** Phân khúc cực kỳ nhiều lượng tương tác.

In [ ]:
model_data <- read_csv('../../data/airbnb_model_ready.csv', show_col_types = FALSE)

features <- model_data %>% select(price, latitude, longitude, number_of_reviews) %>% na.omit()
scaled_features <- scale(features)

set.seed(42)
kmeans_result <- kmeans(scaled_features, centers = 4, nstart = 10)
features$cluster <- as.factor(kmeans_result$cluster)

ggplot(features, aes(x = longitude, y = latitude, color = cluster)) +
  geom_point(alpha = 0.5, size = 0.5) +
  theme_minimal() +
  labs(title = "Phân khúc phòng Airbnb (K-Means Clustering)")

### 2. Dự đoán giá (Regression - Random Forest)

**Kết luận:** Mô hình Random Forest đã học được quy luật định giá với độ lỗi tuyệt đối trung bình (MAE) rơi vào khoảng **$109**. Mức giá AI gợi ý sẽ chênh lệch khoảng 109 USD so với giá thực tế của thị trường. Đây là con số chấp nhận được cho Phase 1.

In [ ]:
set.seed(42)
df_reg <- model_data %>% 
  select(price, room_type, neighbourhood_group, accommodates, minimum_nights, has_reviews) %>%
  mutate(across(c(room_type, neighbourhood_group), as.factor)) %>%
  sample_n(10000)

train_idx <- sample(1:nrow(df_reg), 0.8 * nrow(df_reg))
train_data <- df_reg[train_idx, ]
test_data <- df_reg[-train_idx, ]

rf_model <- randomForest(price ~ ., data = train_data, ntree = 50, importance = TRUE)
predictions <- predict(rf_model, test_data)

mae <- mean(abs(test_data$price - predictions))
cat("Mean Absolute Error (MAE):", mae, "\n")

rss <- sum((predictions - test_data$price) ^ 2)
tss <- sum((test_data$price - mean(test_data$price)) ^ 2)
cat("R-squared (R2):", 1 - rss/tss, "\n")

### 3. Đánh giá Thái độ khách hàng (Sentiment Analysis)

**Mô tả:** Bài toán thứ ba là sử dụng từ điển cảm xúc (`syuzhet`) để chấm điểm tích cực/tiêu cực cho từng bình luận của khách hàng.
**Kết luận:** Phần lớn khách hàng có trải nghiệm tích cực (Sentiment Score > 0). Lượng bình luận tiêu cực chiếm tỷ lệ rất nhỏ. Điều này minh chứng Airbnb tại New York duy trì được chất lượng dịch vụ tốt và tạo được sự hài lòng lớn cho khách lưu trú.

In [ ]:
# Chạy phân tích cảm xúc
# (Nếu báo thiếu gói syuzhet, bạn nhớ chạy lệnh install.packages("syuzhet") trong R Console trước nhé)
library(syuzhet)

cat("Đang đọc dữ liệu reviews đã làm sạch...\n")
reviews <- read_csv('../../data/cleaned_reviews.csv.xz', show_col_types = FALSE)

# Lấy ngẫu nhiên 5000 đánh giá để chạy thử cho nhanh
set.seed(42)
sample_reviews <- reviews %>% 
  filter(!is.na(comments)) %>%
  sample_n(5000)

cat("Đang chấm điểm Sentiment...\n")
sentiment_scores <- get_sentiment(sample_reviews$comments, method = "syuzhet")

sample_reviews$sentiment_score <- sentiment_scores
sample_reviews <- sample_reviews %>%
  mutate(sentiment_label = case_when(
    sentiment_score > 0 ~ "Positive",
    sentiment_score < 0 ~ "Negative",
    TRUE ~ "Neutral"
  ))

cat("\nHoàn thành! Thống kê kết quả:\n")
print(table(sample_reviews$sentiment_label))

# Vẽ biểu đồ bar chart đơn giản cho Sentiment
ggplot(sample_reviews, aes(x = sentiment_label, fill = sentiment_label)) +
  geom_bar(color = "black") +
  theme_minimal() +
  scale_fill_manual(values = c("Negative" = "tomato", "Neutral" = "gray", "Positive" = "palegreen")) +
  labs(title = "Phân bố cảm xúc của khách hàng (Sentiment Analysis)", x = "Cảm xúc", y = "Số lượng bình luận")